In [1]:
import duckdb
import pandas as pd
from pathlib import Path

In [2]:
# Paths
RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")

con = duckdb.connect()

print("DuckDB version:", duckdb.__duckdb_version__)

DuckDB version: 1.5.3


In [7]:
TABLES = {
    "application_train":    RAW / "application_train.csv",
    "application_test":     RAW / "application_test.csv",
    "bureau":               RAW / "bureau.csv",
    "bureau_balance":       RAW / "bureau_balance.csv",
    "previous_application": RAW / "previous_application.csv",
    "POS_CASH_balance":     RAW / "POS_CASH_balance.csv",
    "installments_payments":RAW / "installments_payments.csv",
    "credit_card_balance":  RAW / "credit_card_balance.csv",

}

In [8]:
for name, path in TABLES.items():
    if name == "HomeCredit_columns_description":
        read_expr = f"read_csv('{path}', encoding='latin1', ignore_errors=true)"
    else:
        read_expr = f"'{path}'"

    result = con.execute(f"""
        SELECT COUNT(*) AS rows
        FROM {read_expr}
    """).df()

    rows = result["rows"].iloc[0]

    if name == "HomeCredit_columns_description":
        cols_df = con.execute(f"DESCRIBE SELECT * FROM read_csv('{path}', encoding='latin1', ignore_errors=true)").df()
    else:
        cols_df = con.execute(f"DESCRIBE SELECT * FROM '{path}'").df()

    n_cols = len(cols_df)

    print(f"{'─'*55}")
    print(f"  {name}")
    print(f"  Rows: {rows:,}   Columns: {n_cols}")
    print(f"  Cols: {', '.join(cols_df['column_name'].tolist()[:6])} ...")

───────────────────────────────────────────────────────
  application_train
  Rows: 307,511   Columns: 122
  Cols: SK_ID_CURR, TARGET, NAME_CONTRACT_TYPE, CODE_GENDER, FLAG_OWN_CAR, FLAG_OWN_REALTY ...
───────────────────────────────────────────────────────
  application_test
  Rows: 48,744   Columns: 121
  Cols: SK_ID_CURR, NAME_CONTRACT_TYPE, CODE_GENDER, FLAG_OWN_CAR, FLAG_OWN_REALTY, CNT_CHILDREN ...
───────────────────────────────────────────────────────
  bureau
  Rows: 1,716,428   Columns: 17
  Cols: SK_ID_CURR, SK_ID_BUREAU, CREDIT_ACTIVE, CREDIT_CURRENCY, DAYS_CREDIT, CREDIT_DAY_OVERDUE ...
───────────────────────────────────────────────────────
  bureau_balance
  Rows: 27,299,925   Columns: 3
  Cols: SK_ID_BUREAU, MONTHS_BALANCE, STATUS ...
───────────────────────────────────────────────────────
  previous_application
  Rows: 1,670,214   Columns: 37
  Cols: SK_ID_PREV, SK_ID_CURR, NAME_CONTRACT_TYPE, AMT_ANNUITY, AMT_APPLICATION, AMT_CREDIT ...
───────────────────────────────

In [ ]:
target_dist = con.execute(f"""
    SELECT TARGET,
           COUNT(*) AS count,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM '{RAW}/application_train.csv'
    GROUP BY TARGET
    ORDER BY TARGET
""").df()

print(target_dist)

In [ ]:
for name, path in TABLES.items():
    out = PROCESSED / f"{name}.parquet"
    con.execute(f"""
        COPY (SELECT * FROM '{path}')
        TO '{out}'
        (FORMAT PARQUET, COMPRESSION SNAPPY)
    """)
    print(f"✓  {name}.parquet")

print("\nAll files converted.")

In [ ]:
print(f"{'Table':<30} {'Parquet size':>14} {'Rows':>12}")
print("─" * 58)
for name in TABLES:
    out = PROCESSED / f"{name}.parquet"
    size_mb = out.stat().st_size / 1_048_576
    rows = con.execute(f"SELECT COUNT(*) FROM '{out}'").fetchone()[0]
    print(f"{name:<30} {size_mb:>11.1f} MB {rows:>12,}")